# 🧬 Genome utilities

Random genome generation (`generate_genome`, `generate_genome_with_repeats`) and shotgun read sampling (`generate_reads`).

> **Self-contained.** No imports from other project notebooks.


In [ ]:
import random


#### `generate_genome`
*Generate a purely random DNA sequence (the 'ground truth' genome).*


In [ ]:
def generate_genome(length: int) -> str:
    """Generate a purely random DNA sequence (the 'ground truth' genome).

    Args:
        length: Total length G of the reference genome.
    Returns:
        Random DNA string of the requested length.
    """
    bases = ["A", "C", "G", "T"]
    return "".join(random.choice(bases) for _ in range(length))


#### `generate_genome_with_repeats`
*Generate a DNA sequence with realistic repeat structure.*


In [ ]:
def generate_genome_with_repeats(length: int) -> str:
    """Generate a DNA sequence with realistic repeat structure.

    Simulates LINE, SINE, LTR, and DNA transposon families plus tandem repeats,
    mimicking the composition of real eukaryotic genomes.

    Args:
        length: Total length G of the generated genome.
    Returns:
        DNA string of the requested length containing repeat elements.
    """
    bases = ["A", "C", "G", "T"]

    def gen_random(l):
        return "".join(random.choice(bases) for _ in range(l))

    def mutate(seq, snp_rate=0.1):
        return "".join(
            random.choice(bases) if random.random() < snp_rate else b
            for b in seq
        )

    # Build repeat-element libraries
    repeat_library = {
        "LINE": [gen_random(6000) for _ in range(5)],
        "SINE": [gen_random(300)  for _ in range(10)],
        "LTR":  [gen_random(2000) for _ in range(5)],
        "DNA":  [gen_random(2500) for _ in range(3)],
    }

    weights = [
        ("LINE",   0.20),
        ("SINE",   0.13),
        ("LTR",    0.08),
        ("DNA",    0.03),
        ("TANDEM", 0.05),
        ("UNIQUE", 0.51),
    ]

    def pick_type():
        r, cumulative = random.random(), 0
        for t, w in weights:
            cumulative += w
            if r < cumulative:
                return t

    genome, current_len = [], 0
    while current_len < length:
        t = pick_type()
        if t in repeat_library:
            seq = mutate(random.choice(repeat_library[t]))
        elif t == "TANDEM":
            unit = gen_random(random.randint(1, 6))
            seq  = unit * random.randint(5, 20)
        else:
            seq = gen_random(100)
        genome.append(seq)
        current_len += len(seq)

    return "".join(genome)[:length]


# =============================================================================
# MODULE: read generation
# =============================================================================


#### `generate_reads`
*Simulate shotgun sequencing by sampling random substrings.*


In [ ]:
def generate_reads(genome: str, num_reads: int,
                   min_len: int, max_len: int) -> list:
    """Simulate shotgun sequencing by sampling random substrings.

    Args:
        genome:    Full genome sequence G to sample from.
        num_reads: Number of reads N to generate.
        min_len:   Minimum read length.
        max_len:   Maximum read length (used as fixed L for theory).
    Returns:
        List of read sequences.
    """
    reads, genome_len = [], len(genome)
    for _ in range(num_reads):
        read_len = random.randint(min_len, max_len)
        start    = random.randint(0, genome_len - read_len)
        reads.append(genome[start : start + read_len])
    return reads


# =============================================================================
# SIMULATIONS
# =============================================================================
